# EXPERIMENTOS DEL PAPER (versión corregida, octubre 2026)

El código de cada benchmark está en un archivo `.py` propio (`advection_nas.py`, `poisson_nas.py`) que debe estar **en la misma carpeta que este notebook**. Así las funciones de advección y de Poisson, que se llaman igual, no se pisan.

Ejecuta las celdas en orden: **IMPORTS → CONFIG → RUN ALL → RESUMEN**. Nada más.

- **IMPORTS**: carga los dos módulos como `ADV` y `POI`.
- **CONFIG**: semillas, learning rates y rutas. `LR_ALPHA = 1e-2` es el cambio clave (antes 1e-4, con lo que las alphas apenas se movían y el softmax se quedaba uniforme).
- **RUN ALL**: por cada semilla y benchmark: búsqueda → discretización → tres entrenamientos finales (scratch, herencia, herencia+warm-up). Imprime `pi_max por capa` (cuánto se ha concentrado el softmax). Guarda en `results/` al terminar cada semilla: si se corta, no se pierde lo hecho.
- **RESUMEN**: lee `results/`, imprime las tablas y guarda las figuras. Se puede ejecutar en cualquier momento, incluso con RUN ALL a medias.

Si editas un `.py` con el kernel ya arrancado, vuelve a ejecutar IMPORTS (tiene `importlib.reload`) para que el cambio se cargue.


### IMPORTS

In [ ]:
# ===================== IMPORTS =====================
import importlib, sys, os
sys.path.insert(0, os.getcwd())          # para que Python encuentre los .py de esta carpeta
import torch, numpy as np
import advection_nas as ADV
import poisson_nas as POI
importlib.reload(ADV); importlib.reload(POI)   # recarga si has editado los .py
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("ADV:", ADV.run_advection_seed_experiment.__name__, "| POI:", POI.run_poisson_seed_experiment.__name__)


### CONFIG

In [ ]:
# ===================== CONFIG =====================
import os, json, time
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]      # 5 semillas (antes 3)
LR_ALPHA = 1e-2              # antes 1e-4 -> el softmax no podía concentrarse
SEARCH_MAX_ITERS = 1500      # tope de la búsqueda; para antes si la arquitectura es estable STABLE_ITERS iters
STABLE_ITERS = 300
TOTAL_ITERS = 5000           # entrenamiento final, igual para los tres métodos
WARMUP_ITERS, WARMUP_LR = 100, 5e-5
MSE_THRESHOLD = 1e-5
RUN_ADVECTION, RUN_POISSON = True, True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, torch.Tensor): return o.detach().cpu().tolist()
    return o

def save_json(obj, name):
    with open(os.path.join(RESULTS_DIR, name), "w") as f:
        json.dump(_jsonable(obj), f)


### RUN ALL

In [ ]:
# ===================== RUN ALL =====================
x_eval = torch.linspace(-1, 1, 200).reshape(-1, 1)
xt_list, tt_list = [], []
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    xt_list.append(x_eval); tt_list.append(torch.ones_like(x_eval) * t)
x_eval_all = torch.cat(xt_list, dim=0); t_eval_all = torch.cat(tt_list, dim=0)

METHODS = ["scratch", "inheritance", "inheritance_warmup"]
advection_seed_summaries, advection_all_results = [], {}
poisson_seed_summaries, poisson_all_results = [], {}
t0 = time.time()

def _line(summary):
    return (f"  arch={summary['architecture']}  "
            f"MSE scratch={summary['scratch']['final_mse']:.2e}  "
            f"inh={summary['inheritance']['final_mse']:.2e}  "
            f"inh+wu={summary['inheritance_warmup']['final_mse']:.2e}  "
            f"| loss inicial scratch={summary['scratch']['initial_loss']:.2e} inh={summary['inheritance']['initial_loss']:.2e}  "
            f"[{(time.time()-t0)/60:.1f} min]")

for seed in SEEDS:
    if RUN_ADVECTION:
        print(f"\n########## ADVECTION seed {seed} ##########")
        summary, results = ADV.run_advection_seed_experiment(
            seed=seed, x_eval_all=x_eval_all, t_eval_all=t_eval_all, device=device,
            resample_every_search=100, resample_every_final=100,
            n_interior=2000, n_initial=200, n_boundary=100,
            search_max_iterations=SEARCH_MAX_ITERS, total_iterations=TOTAL_ITERS,
            stable_iters=STABLE_ITERS,
            lr_weights_search=1e-3, lr_weights_train=1e-4, lr_alpha=LR_ALPHA,
            max_layers=6, neuron_options=[25, 50, 75, 100],
            warmup_iterations=WARMUP_ITERS, warmup_lr=WARMUP_LR,
            mse_threshold=MSE_THRESHOLD, eval_every=50, silent=True)
        advection_seed_summaries.append(summary); advection_all_results[seed] = results
        save_json(advection_seed_summaries, "advection_summaries.json")
        save_json({s: {m: r[m]["history"]["losses"] for m in METHODS} for s, r in advection_all_results.items()},
                  "advection_losses.json")
        print(_line(summary))

    if RUN_POISSON:
        print(f"\n########## POISSON seed {seed} ##########")
        summary, results = POI.run_poisson_seed_experiment(
            seed=seed, device=device, activation='tanh',
            resample_every_search=1, resample_every_final=100,
            n_interior=5000, n_boundary=2000,
            search_iterations=SEARCH_MAX_ITERS, total_iterations=TOTAL_ITERS,
            stable_iters=STABLE_ITERS,
            lr_weights_search=1e-3, lr_weights_train=1e-4, lr_alpha=LR_ALPHA,
            warmup_iterations=WARMUP_ITERS, warmup_lr=WARMUP_LR,
            mse_threshold=MSE_THRESHOLD, eval_every=50, silent=True)
        poisson_seed_summaries.append(summary); poisson_all_results[seed] = results
        save_json(poisson_seed_summaries, "poisson_summaries.json")
        save_json({s: {m: r[m]["history"]["losses"] for m in METHODS} for s, r in poisson_all_results.items()},
                  "poisson_losses.json")
        print(_line(summary))

print(f"\nTODO TERMINADO en {(time.time()-t0)/60:.1f} min. Resultados en ./{RESULTS_DIR}/")


### RESUMEN

In [ ]:
# ===================== RESUMEN (tablas + figuras) =====================
import pandas as pd, matplotlib.pyplot as plt

def load(name):
    p = os.path.join(RESULTS_DIR, name)
    return json.load(open(p)) if os.path.exists(p) else None

rows, pi_rows = [], []
for bench in ["advection", "poisson"]:
    S = load(f"{bench}_summaries.json")
    if not S: continue
    for run in S:
        pis = [p for _, _, p in run["pi_max"]]
        pi_rows.append({"benchmark": bench, "seed": run["seed"], "architecture": str(run["architecture"]),
                        "pi_min": min(pis), "pi_mean": float(np.mean(pis)),
                        "inh_initial_loss": run["inheritance"]["initial_loss"],
                        "scratch_initial_loss": run["scratch"]["initial_loss"],
                        "inh_final_mse": run["inheritance"]["final_mse"],
                        "scratch_final_mse": run["scratch"]["final_mse"]})
        for m in ["scratch", "inheritance", "inheritance_warmup"]:
            h = run[m]
            rows.append({"benchmark": bench, "seed": run["seed"], "method": m,
                         "initial_loss": h["initial_loss"], "final_loss": h["final_loss"],
                         "final_mse": h["final_mse"], "threshold_reached": int(h["threshold_reached"]),
                         "iterations_to_threshold": h["iterations_to_threshold"]})

df = pd.DataFrame(rows); pi_df = pd.DataFrame(pi_rows)
pd.set_option("display.width", 220); pd.set_option("display.float_format", lambda v: f"{v:.3e}")

print("=== TABLA 1: por benchmark y método (media ± std sobre semillas) ===")
agg = df.groupby(["benchmark", "method"]).agg(
    initial_loss_mean=("initial_loss","mean"), initial_loss_std=("initial_loss","std"),
    final_loss_mean=("final_loss","mean"),     final_loss_std=("final_loss","std"),
    final_mse_mean=("final_mse","mean"),       final_mse_std=("final_mse","std"),
    success=("threshold_reached","sum"), n=("threshold_reached","count"),
    iters_to_thr=("iterations_to_threshold","mean"))
print(agg.to_string()); print()
print("=== TABLA 2: concentración del softmax por semilla (clave para la Sec. 3.4 del paper) ===")
print(pi_df.to_string(index=False)); print()

df.to_csv(os.path.join(RESULTS_DIR, "per_run_metrics.csv"), index=False)
agg.to_csv(os.path.join(RESULTS_DIR, "aggregated_metrics.csv"))
pi_df.to_csv(os.path.join(RESULTS_DIR, "softmax_concentration.csv"), index=False)

labels = {"scratch": "Scratch", "inheritance": "Inheritance", "inheritance_warmup": "Inheritance + Warm-up"}
for bench in ["advection", "poisson"]:
    L = load(f"{bench}_losses.json")
    if not L: continue
    for seed, hist in L.items():
        plt.figure(figsize=(8, 5))
        for m in labels:
            plt.plot(np.arange(1, len(hist[m]) + 1), hist[m], label=labels[m], linewidth=2)
        plt.yscale("log"); plt.xlabel("Iteration"); plt.ylabel("Training loss")
        plt.title(f"{bench.capitalize()} benchmark (seed {seed}): convergence")
        plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, f"{bench}_seed{seed}_convergence.png"), dpi=300, bbox_inches="tight"); plt.close()
print("Figuras guardadas:", sorted(f for f in os.listdir(RESULTS_DIR) if f.endswith(".png")))
print("\nMÁNDAME la carpeta results/ entera (csv + json + png).")
